| Concept   | On regarde quoi ?      | Signification                      |
| --------- | ---------------------- | ---------------------------------- |
| SASA      | Surface brute (Å²)     | Surface accessible                 |
| rSASA     | Surface relative (0-1) | % exposition                       |
| Surface   | rSASAm > 0.25          | Résidu exposé                      |
| Interface | ΔrSASA > 0.01          | Résidu impliqué dans l’interaction |


In [1]:
import pickle
import pandas as pd


# 1- Importing docs

In [2]:
m1_file = "2FJU_AB_m1.pkl"
m2_file = "2FJU_AB_m2.pkl"
c_file  = "2FJU_AB_c.pkl"

d_m1 = pickle.load(open(m1_file, "rb"))
d_m2 = pickle.load(open(m2_file, "rb"))
d_c  = pickle.load(open(c_file,  "rb"))

# 2- Useful functions

In [3]:
def get_chain_keys(d):
    """Return chain keys (exclude 'chains' if present)."""
    return [k for k in d.keys() if k != "chains"]

def guess_chain_map(d_m, d_c, fallback_chain):
    """
    Try to guess the chain mapping between monomer and complex by matching chain keys.
    If no match is found, return the fallback chain.
    """
    m_chains = get_chain_keys(d_m)
    c_chains = get_chain_keys(d_c)

    if len(m_chains) == 0:
        raise ValueError("No chains found in monomer.")
    if len(c_chains) == 0:
        raise ValueError("No chains found in complex.")

    cm = m_chains[0]
    if cm in c_chains:
        return cm
    return fallback_chain

def extract_residue_ids(d_chain):
    """List of residue IDs (keys) excluding 'reslist' if present."""
    return [r for r in d_chain.keys() if r != "reslist"]

def build_table(d_m, d_c, chain_m, chain_c, monomer_name, thr_surface=0.25, thr_interface=0.01):
    """
    Build a DataFrame comparing residues in monomer and complex for given chains.
    Classify residues as surface and interface based on rSASA values and thresholds.
    """
    rows = []
    res_ids = extract_residue_ids(d_m[chain_m])

    for r in res_ids:
        #Security: skip if residue not in complex data
        if chain_c not in d_c or r not in d_c[chain_c]:
            continue

        rm = d_m[chain_m][r]
        rc = d_c[chain_c][r]
        
        rSASAm = rm.get("rSASA", None)
        SASAm  = rm.get("SASA", None)
        rSASAc = rc.get("rSASA", None)
        SASAc  = rc.get("SASA", None)

        # type of residue: try multiple keys, default to "NA"
        res_type = (rm.get("resname")
                    or rm.get("AA")
                    or rm.get("type")
                    or rm.get("name")
                    or "NA")

        # if rSASA is missing in either monomer or complex, skip this residue
        if rSASAm is None or rSASAc is None:
            continue

        delta = rSASAm - rSASAc
        is_surface   = (rSASAm > thr_surface)
        is_interface = (delta  > thr_interface)

        rows.append({
            "monomer": monomer_name,
            "chain_monomer": chain_m,
            "chain_complex": chain_c,
            "res_id": r,
            "res_type": res_type,
            "rSASAm": rSASAm,
            "SASAm": SASAm,
            "rSASAc": rSASAc,
            "SASAc": SASAc,
            "delta_rSASA": delta,
            "is_surface": is_surface,
            "is_interface": is_interface
        })

    return pd.DataFrame(rows)


# 3- Chosing chains

In [4]:
print("Monomère 1 chains:", get_chain_keys(d_m1))
print("Monomère 2 chains:", get_chain_keys(d_m2))
print("Complexe chains  :", get_chain_keys(d_c))

Monomère 1 chains: ['B']
Monomère 2 chains: ['A']
Complexe chains  : ['B', 'A']


In [5]:
# if it doesn't work, we can always force m1->A and m2->B
chain_c_m1 = guess_chain_map(d_m1, d_c, fallback_chain="A")
chain_c_m2 = guess_chain_map(d_m2, d_c, fallback_chain="B")

chain_m1 = get_chain_keys(d_m1)[0]
chain_m2 = get_chain_keys(d_m2)[0]

print("Mapping choisi:")
print("m1:", chain_m1, "-> complex:", chain_c_m1)
print("m2:", chain_m2, "-> complex:", chain_c_m2)


Mapping choisi:
m1: B -> complex: B
m2: A -> complex: A


# 4- Table construction

In [6]:
df1 = build_table(d_m1, d_c, chain_m1, chain_c_m1, monomer_name="m1")
df2 = build_table(d_m2, d_c, chain_m2, chain_c_m2, monomer_name="m2")

df = pd.concat([df1, df2], ignore_index=True)

In [7]:
df

,monomer,chain_monomer,chain_complex,res_id,res_type,rSASAm,SASAm,rSASAc,SASAc,delta_rSASA,is_surface,is_interface
0,m1,B,B,11,PRO,0.714692,98.062944,0.714692,98.062944,0.0,True,False
1,m1,B,B,12,LYS,0.919977,188.576870,0.919977,188.576870,0.0,True,False
2,m1,B,B,13,VAL,0.089422,13.589396,0.089422,13.589396,0.0,False,False
3,m1,B,B,14,LYS,0.554962,113.756053,0.554962,113.756053,0.0,True,False
4,m1,B,B,15,ALA,0.776759,84.480317,0.776759,84.480317,0.0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
868,m2,A,A,173,ILE,0.000616,0.108192,0.000616,0.108192,0.0,False,False
869,m2,A,A,174,ARG,0.439235,104.612586,0.439235,104.612586,0.0,True,False
870,m2,A,A,175,ALA,0.293903,31.964839,0.293903,31.964839,0.0,True,False
871,m2,A,A,176,VAL,0.263810,40.091185,0.263810,40.091185,0.0,True,False


In [8]:
df["delta_rSASA"].unique()

array([0.00000000e+00, 3.71531519e-01, 1.45934146e-01, 2.20252678e-01,
       3.04545358e-01, 3.68623408e-02, 7.18944635e-02, 8.94374722e-04,
       7.59456402e-01, 1.36414120e-01, 1.89616931e-01, 1.53890327e-01,
       4.14264301e-01, 2.21627822e-02, 2.51967938e-01, 1.67849945e-02,
       5.60999531e-03, 3.98484265e-01, 4.11428585e-02, 4.56591493e-02,
       3.62326063e-01, 6.29786084e-01, 1.46150861e-01, 2.59375019e-01,
       9.25745907e-05, 2.85220241e-01, 4.70733706e-02, 1.85200614e-01,
       1.17373780e-02, 7.40118406e-02, 4.27939976e-01, 4.14170264e-01,
       5.78327643e-01, 1.62255854e-01])

In [9]:
print(df["monomer"].unique())
print(df["chain_complex"].unique())
print(df["chain_monomer"].unique())


['m1' 'm2']
['B' 'A']
['B' 'A']


# 5- Results

In [12]:
surface_df   = df[df["is_surface"]].copy()
interface_df = df[df["is_interface"]].copy()

print("\n--- Resume ---")
print("Total résidus analysés:", len(df))
print("Résidus SURFACE:", len(surface_df))
print("Résidus INTERFACE:", len(interface_df))



--- Resume ---
Total résidus analysés: 873
Résidus SURFACE: 385
Résidus INTERFACE: 30


In [13]:
print("\nExample interface (top 10 par ΔrSASA):")
print(interface_df.sort_values("delta_rSASA", ascending=False).head(10)[["monomer","chain_complex","res_id","res_type","rSASAm","rSASAc","delta_rSASA"]])


Example interface (top 10 par ΔrSASA):
    monomer chain_complex res_id res_type    rSASAm    rSASAc  delta_rSASA
41       m1             B     52      GLN  0.842726  0.083270     0.759456
732      m2             A     37      PHE  0.676342  0.046556     0.629786
765      m2             A     70      LEU  0.747868  0.169540     0.578328
761      m2             A     66      ARG  0.924683  0.496743     0.427940
73       m1             B     84      VAL  0.617911  0.203646     0.414264
762      m2             A     67      LEU  0.591524  0.177354     0.414170
107      m1             B    118      TYR  0.533494  0.135010     0.398484
11       m1             B     22      ARG  0.601419  0.229888     0.371532
731      m2             A     36      VAL  0.578879  0.216553     0.362326
24       m1             B     35      PRO  0.366277  0.061732     0.304545


In [15]:
print("\nExample surface (top 10 par ΔrSASA):")
print(surface_df.sort_values("delta_rSASA", ascending=True).head(10)[["monomer","chain_complex","res_id","res_type","rSASAm","rSASAc","delta_rSASA"]])


Example surface (top 10 par ΔrSASA):
    monomer chain_complex res_id res_type    rSASAm    rSASAc  delta_rSASA
190      m1             B    201      GLU  0.730393  0.730393          0.0
843      m2             A    148      GLU  0.559326  0.559326          0.0
845      m2             A    150      GLY  0.618038  0.618038          0.0
847      m2             A    152      VAL  0.498934  0.498934          0.0
848      m2             A    153      LYS  0.482370  0.482370          0.0
855      m2             A    160      LEU  0.555809  0.555809          0.0
856      m2             A    161      THR  0.579682  0.579682          0.0
857      m2             A    162      GLN  0.294358  0.294358          0.0
858      m2             A    163      ARG  0.691034  0.691034          0.0
859      m2             A    164      GLY  0.343444  0.343444          0.0


In [14]:
interface_df

,monomer,chain_monomer,chain_complex,res_id,res_type,rSASAm,SASAm,rSASAc,SASAc,delta_rSASA,is_surface,is_interface
11,m1,B,B,22,ARG,0.601419,143.240021,0.229888,54.752359,0.371532,True,True
13,m1,B,B,24,ILE,0.147694,25.954347,0.001760,0.309339,0.145934,False,True
22,m1,B,B,33,ALA,0.303449,33.003163,0.083197,9.048482,0.220253,True,True
24,m1,B,B,35,PRO,0.366277,50.256874,0.061732,8.470205,0.304545,True,True
25,m1,B,B,36,VAL,0.037076,5.634380,0.000213,0.032410,0.036862,False,True
26,m1,B,B,37,ILE,0.300892,52.875782,0.228998,40.241768,0.071894,True,True
41,m1,B,B,52,GLN,0.842726,150.704737,0.083270,14.891149,0.759456,True,True
42,m1,B,B,53,SER,0.613184,72.564216,0.476770,56.420969,0.136414,True,True
43,m1,B,B,54,LYS,0.792500,162.446660,0.602883,123.578982,0.189617,True,True
72,m1,B,B,83,ASP,0.597458,85.293142,0.443568,63.323759,0.153890,True,True


# 6- CSV

In [16]:
df.to_csv("all_residues_surface_interface.csv", index=False)
surface_df.to_csv("surface_residues.csv", index=False)
interface_df.to_csv("interface_residues.csv", index=False)

print("\nCSV générés:")
print("- all_residues_surface_interface.csv")
print("- surface_residues.csv")
print("- interface_residues.csv")


CSV générés:
- all_residues_surface_interface.csv
- surface_residues.csv
- interface_residues.csv
